# Generative Sequential Recommendation — Colab 训练 (T5 + RQ-VAE Semantic ID)

**运行前**：Runtime → Change runtime type → **GPU**（推荐 A100）

**架构**：T5 encoder-decoder（对齐 TIGER 参考实现），从头训练，~4.6M 参数。
历史序列拍平成 Semantic ID token 输入 encoder，decoder 自回归生成下一个 item 的 4 个 token。

**需要手动上传**：
- `embedding/semantic_ids_rqvae.npy`（本地已生成，~170 KB）

`data/beauty_data.pkl` 已在 GitHub 仓库中，clone 后自动存在。

本 notebook 只跑下游 T5 训练 + 评估，不重新训 RQ-VAE。
如需重训 RQ-VAE，参考 `notebooks/colab_rqvae.ipynb`。

---

## 运行顺序
1. **环境**（Cell 1–4）：GPU 检查 → 装依赖 → clone repo → 上传 semantic IDs
2. **训练 + 评估**（Cell 5–6）：`train.py` → `evaluate.py`
3. **下载结果**（Cell 7）

In [ ]:
# Cell 1：确认 GPU 可用
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请先到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：克隆仓库
!git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
%cd Generative-Sequential-Recommendation
!pwd && ls

In [ ]:
# Cell 4：上传 semantic_ids_rqvae.npy
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('请上传 semantic_ids_rqvae.npy（本地项目 embedding/ 目录里）')
uploaded = files.upload()
for fname in uploaded:
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    print(f'✅ 已移动到 {dest}  ({os.path.getsize(dest)/1024:.0f} KB)')

import numpy as np
sids = np.load('embedding/semantic_ids_rqvae.npy', allow_pickle=True).item()
vals = list(sids.values())
print(f'\n✅ 加载成功：{len(sids)} items')
print(f'   示例：item {list(sids.keys())[0]} → {vals[0]}')
assert len(vals[0]) == 4, f'期望 4-token ID，实际 {len(vals[0])}'

ranges = list(zip(*vals))
for i, r in enumerate(ranges):
    print(f'   c{i+1} 范围: {min(r)}~{max(r)}')

unique_ids = len(set(tuple(v) for v in vals))
print(f'   唯一 4-tuple 数: {unique_ids} / {len(sids)}  {"✓ 零冲突" if unique_ids == len(sids) else "✗ 有冲突！"}')

---
## 训练生成式推荐模型（T5 encoder-decoder + RQ-VAE 4-token Semantic ID）

In [ ]:
# Cell 5：训练
# 200 epoch + 全量 val 每 2 epoch 评估，A100 约 6-8 小时
# 常数 lr=1e-4，对齐 TIGER 参考实现（无 scheduler）
# 早停依据 validation NDCG@10
# 输出：checkpoints/best_model_t5.pt
!python train.py

In [ ]:
# Cell 6：评估（all-rank Recall@K / NDCG@K，constrained beam search）
!python evaluate.py

---
## 下载结果

In [ ]:
# Cell 7：下载 checkpoint
import os
from google.colab import files

fpath = 'checkpoints/best_model_t5.pt'
if os.path.exists(fpath):
    files.download(fpath)
    print(f'✅ 下载：{fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
else:
    print(f'⚠️  {fpath} 不存在')